# Week 4 - Day 2 : Daily Challenge
# Integration NumPy, Pandas & Matplotlib
# Analyse de la Base de Donnees Mondiale des Centrales Electriques (WRI)

**Source :** World Resources Institute — Global Power Plant Database  
**Objectif :** Analyser les centrales electriques mondiales : capacite, carburant, geographie et evolution temporelle.

## 0. Setup & Chargement des donnees

In [ ]:
!pip install seaborn scipy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# Telechargement du dataset WRI
import urllib.request, os

URL  = 'https://raw.githubusercontent.com/wri/global-power-plant-database/master/output_database/global_power_plant_database.csv'
FILE = 'global_power_plant_database.csv'

if not os.path.exists(FILE):
    print('Telechargement du dataset...')
    urllib.request.urlretrieve(URL, FILE)
    print('Telechargement termine.')
else:
    print('Dataset deja present.')

df_raw = pd.read_csv(FILE)
print(f'Shape : {df_raw.shape}')
df_raw.head(3)

---
## 1. Importation et Nettoyage des Donnees

In [ ]:
# Colonnes utiles
cols = [
    'country', 'country_long', 'name', 'capacity_mw',
    'latitude', 'longitude', 'primary_fuel',
    'commissioning_year',
    'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019'
]
df = df_raw[cols].copy()

# Valeurs manquantes avant nettoyage
print('=== Valeurs manquantes avant nettoyage ===')
print(df.isnull().sum())

# Conversion NumPy des colonnes numeriques
num_cols = ['capacity_mw', 'latitude', 'longitude',
            'commissioning_year', 'generation_gwh_2017',
            'generation_gwh_2018', 'generation_gwh_2019']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].astype(np.float64)

# Supprimer les lignes sans capacite (colonne cle)
df = df.dropna(subset=['capacity_mw'])

# Remplir annee manquante par la mediane
median_year = np.nanmedian(df['commissioning_year'].values)
df['commissioning_year'] = df['commissioning_year'].fillna(median_year)
df['commissioning_year'] = df['commissioning_year'].astype(int)

# Remplir generation manquante par 0
for col in ['generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019']:
    df[col] = df[col].fillna(0)

# Filtrer les annees aberrantes
df = df[(df['commissioning_year'] >= 1900) & (df['commissioning_year'] <= 2023)]

print(f'\nShape apres nettoyage : {df.shape}')
print(f'Valeurs manquantes restantes :\n{df.isnull().sum()}')

---
## 2. Analyse Exploratoire des Donnees (EDA)

In [ ]:
# Statistiques cles avec Pandas
print('=== Statistiques descriptives (capacite_mw) ===')
cap = df['capacity_mw'].values  # tableau NumPy
print(f'Nombre de centrales : {len(cap):,}')
print(f'Moyenne             : {np.mean(cap):.2f} MW')
print(f'Mediane             : {np.median(cap):.2f} MW')
print(f'Ecart-type          : {np.std(cap):.2f} MW')
print(f'Min                 : {np.min(cap):.2f} MW')
print(f'Max                 : {np.max(cap):.2f} MW')
print(f'Capacite totale     : {np.sum(cap)/1000:.0f} GW')

In [ ]:
# Top 10 pays par nombre de centrales
print('=== Top 10 pays par nombre de centrales ===')
top_pays = df['country_long'].value_counts().head(10)
print(top_pays)

print('\n=== Top 10 pays par capacite totale (GW) ===')
top_cap = df.groupby('country_long')['capacity_mw'].sum().sort_values(ascending=False).head(10) / 1000
print(top_cap.round(1))

In [ ]:
# Distribution par type de carburant
print('=== Distribution par type de carburant ===')
fuel_count = df['primary_fuel'].value_counts()
fuel_cap   = df.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False) / 1000

print('\nNombre de centrales par carburant :')
print(fuel_count)
print('\nCapacite totale par carburant (GW) :')
print(fuel_cap.round(1))

---
## 3. Analyse Statistique (NumPy + SciPy)

In [ ]:
# Analyse de la capacite par type de carburant
top_fuels = df['primary_fuel'].value_counts().head(6).index.tolist()

print('=== Statistiques de capacite par carburant (NumPy) ===')
print(f'{"Carburant":<15} {"N":>6} {"Moyenne":>10} {"Mediane":>10} {"Std":>10} {"Max":>10}')
print('-' * 63)

stats_par_fuel = {}
for fuel in top_fuels:
    vals = df[df['primary_fuel'] == fuel]['capacity_mw'].values
    stats_par_fuel[fuel] = vals
    print(f'{fuel:<15} {len(vals):>6} {np.mean(vals):>10.1f} {np.median(vals):>10.1f} '
          f'{np.std(vals):>10.1f} {np.max(vals):>10.1f}')

In [ ]:
# Test d'hypothese : ANOVA one-way (capacite differe-t-elle selon le carburant ?)
groupes = [stats_par_fuel[f] for f in top_fuels]
f_stat, p_value = stats.f_oneway(*groupes)

print('=== Test ANOVA one-way ===')
print(f'H0 : La capacite moyenne est identique pour tous les types de carburant')
print(f'H1 : Au moins un type de carburant a une capacite moyenne differente')
print(f'\nF-statistique : {f_stat:.4f}')
print(f'p-value       : {p_value:.6f}')
print(f'\nConclusion : {"On REJETTE H0" if p_value < 0.05 else "On NE rejette PAS H0"} (seuil alpha=0.05)')
if p_value < 0.05:
    print('=> La capacite de sortie differe significativement selon le type de carburant.')

In [ ]:
# Test de Pearson : correlation capacite vs generation 2019
mask = df['generation_gwh_2019'] > 0
x = df.loc[mask, 'capacity_mw'].values
y = df.loc[mask, 'generation_gwh_2019'].values

r, p = stats.pearsonr(x, y)
print('=== Correlation Pearson : Capacite MW vs Generation GWh 2019 ===')
print(f'r = {r:.4f}  |  p-value = {p:.2e}')
print(f'Correlation : {"forte" if abs(r) > 0.7 else "moderee" if abs(r) > 0.4 else "faible"} '
      f'et {"significative" if p < 0.05 else "non significative"}')

---
## 4. Analyse des Series Chronologiques

In [ ]:
# Evolution du nombre de nouvelles centrales par annee
year_counts = df.groupby('commissioning_year').size()
year_cap    = df.groupby('commissioning_year')['capacity_mw'].sum() / 1000

years = np.array(year_counts.index)
counts = np.array(year_counts.values)

# Tendance lineaire NumPy (polyfit)
mask_modern = years >= 1980
coeffs = np.polyfit(years[mask_modern], counts[mask_modern], deg=1)
trend  = np.poly1d(coeffs)

print('=== Tendance du nombre de nouvelles centrales (1980-2023) ===')
print(f'Pente : {coeffs[0]:+.1f} centrales/an')
print(f'=> {"Augmentation" if coeffs[0] > 0 else "Diminution"} du nombre de nouvelles centrales par an')

In [ ]:
# Evolution du mix energetique par decennie
df['decennie'] = (df['commissioning_year'] // 10) * 10
fuels_decennie = df[df['primary_fuel'].isin(top_fuels)].groupby(
    ['decennie', 'primary_fuel'])['capacity_mw'].sum().unstack(fill_value=0)

# Normaliser en pourcentage
fuels_pct = fuels_decennie.div(fuels_decennie.sum(axis=1), axis=0) * 100

print('=== Mix energetique par decennie (% de capacite installee) ===')
print(fuels_pct.round(1).to_string())

---
## 5. Visualisations Avancees

In [ ]:
# Graphique 1 : Top 10 pays par capacite totale
fig, ax = plt.subplots(figsize=(12, 6))

colors = cm.viridis(np.linspace(0.2, 0.9, 10))
bars = ax.barh(top_cap.index[::-1], top_cap.values[::-1], color=colors)

for bar, val in zip(bars, top_cap.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} GW', va='center', fontsize=10)

ax.set_title('Top 10 pays par capacite totale installee (GW)', fontsize=15, fontweight='bold')
ax.set_xlabel('Capacite totale (GW)', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('plot1_top10_pays.png', dpi=150)
plt.show()

In [ ]:
# Graphique 2 : Pie chart du mix energetique mondial
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fuel_n   = df['primary_fuel'].value_counts().head(8)
fuel_cap_pie = df.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False).head(8)

palette = plt.cm.Set3(np.linspace(0, 1, 8))

axes[0].pie(fuel_n.values, labels=fuel_n.index, autopct='%1.1f%%',
            colors=palette, startangle=140)
axes[0].set_title('Nombre de centrales par carburant', fontweight='bold')

axes[1].pie(fuel_cap_pie.values, labels=fuel_cap_pie.index, autopct='%1.1f%%',
            colors=palette, startangle=140)
axes[1].set_title('Capacite totale par carburant', fontweight='bold')

plt.suptitle('Mix energetique mondial', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot2_mix_energetique.png', dpi=150)
plt.show()

In [ ]:
# Graphique 3 : Evolution temporelle + tendance
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Nombre de nouvelles centrales
axes[0].bar(years, counts, color='steelblue', alpha=0.6, label='Nouvelles centrales')
x_trend = np.linspace(1980, 2023, 100)
axes[0].plot(x_trend, trend(x_trend), color='red', lw=2, label=f'Tendance (pente={coeffs[0]:+.1f}/an)')
axes[0].set_title('Nombre de nouvelles centrales par annee', fontweight='bold')
axes[0].set_xlabel('Annee')
axes[0].set_ylabel('Nombre de centrales')
axes[0].legend()
axes[0].set_xlim(1950, 2024)
axes[0].grid(alpha=0.3)

# Evolution du mix energetique par decennie
fuels_pct_mod = fuels_pct[fuels_pct.index >= 1960]
fuels_pct_mod.plot(kind='bar', stacked=True, ax=axes[1],
                   colormap='Set2', edgecolor='white')
axes[1].set_title('Evolution du mix energetique par decennie (%)', fontweight='bold')
axes[1].set_xlabel('Decennie')
axes[1].set_ylabel('% de capacite installee')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(loc='upper right', fontsize=9, ncol=2)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot3_series_temporelles.png', dpi=150)
plt.show()

In [ ]:
# Graphique 4 : Carte geographique des centrales (latitude/longitude)
df_geo = df.dropna(subset=['latitude', 'longitude'])
df_geo = df_geo[df_geo['primary_fuel'].isin(top_fuels[:6])]

fig, ax = plt.subplots(figsize=(16, 8))

fuel_colors = {
    'Solar': '#FFD700', 'Wind': '#87CEEB', 'Hydro': '#1E90FF',
    'Gas': '#FF8C00', 'Coal': '#696969', 'Nuclear': '#FF4500'
}

for fuel in top_fuels[:6]:
    if fuel not in fuel_colors:
        continue
    sub = df_geo[df_geo['primary_fuel'] == fuel]
    sizes = np.clip(sub['capacity_mw'].values / 200, 1, 50)
    ax.scatter(sub['longitude'], sub['latitude'],
               c=fuel_colors[fuel], s=sizes, alpha=0.5,
               label=fuel, edgecolors='none')

ax.set_title('Repartition geographique des centrales electriques mondiales',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.axhline(0, color='gray', lw=0.5, linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', lw=0.5, linestyle='--', alpha=0.5)
ax.legend(loc='lower left', fontsize=10, title='Type de carburant')
ax.grid(alpha=0.2)
ax.set_facecolor('#f0f8ff')

plt.tight_layout()
plt.savefig('plot4_carte_geographique.png', dpi=150)
plt.show()

In [ ]:
# Graphique 5 : Boxplot capacite par type de carburant
fig, ax = plt.subplots(figsize=(13, 6))

data_box = [np.log1p(stats_par_fuel[f]) for f in top_fuels]
bp = ax.boxplot(data_box, labels=top_fuels, patch_artist=True, notch=True)

box_colors = cm.Set2(np.linspace(0, 1, len(top_fuels)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_title('Distribution de la capacite par type de carburant (log scale)', fontsize=14, fontweight='bold')
ax.set_xlabel('Type de carburant', fontsize=12)
ax.set_ylabel('log(1 + Capacite en MW)', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot5_boxplot_carburant.png', dpi=150)
plt.show()

In [ ]:
# Graphique 6 : Heatmap de correlation (NumPy + Seaborn)
num_df = df[['capacity_mw', 'latitude', 'longitude',
             'commissioning_year',
             'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019']].dropna()

# Matrice de correlation NumPy
corr_matrix = np.corrcoef(num_df.values.T)
corr_df = pd.DataFrame(corr_matrix, index=num_df.columns, columns=num_df.columns)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
ax.set_title('Matrice de correlation des attributs numeriques', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('plot6_heatmap_correlation.png', dpi=150)
plt.show()

---
## 6. Operations Matricielles (NumPy)

In [ ]:
# Matrice de correlation + Valeurs propres (eigenvalues)
print('=== Analyse par valeurs propres (PCA context) ===')

# Normalisation Z-score (NumPy)
X = num_df.values
X_norm = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

# Matrice de covariance
cov = np.cov(X_norm.T)

# Decomposition en valeurs propres
eigenvalues, eigenvectors = np.linalg.eig(cov)

# Trier par variance expliquee decroissante
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

variance_exp = eigenvalues / np.sum(eigenvalues) * 100
variance_cum = np.cumsum(variance_exp)

print(f'{'Composante':<15} {'Valeur propre':>15} {'Var. expliquee':>15} {'Var. cumulee':>14}')
print('-' * 60)
for i, (ev, ve, vc) in enumerate(zip(eigenvalues, variance_exp, variance_cum)):
    print(f'PC{i+1:<13} {ev:>15.4f} {ve:>14.2f}% {vc:>13.2f}%')

In [ ]:
# Visualisation scree plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

comps = [f'PC{i+1}' for i in range(len(eigenvalues))]

axes[0].bar(comps, variance_exp, color='steelblue', edgecolor='white')
axes[0].set_title('Variance expliquee par composante (Scree Plot)', fontweight='bold')
axes[0].set_ylabel('Variance expliquee (%)')
axes[0].set_xlabel('Composante principale')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(comps, variance_cum, 'o-', color='crimson', lw=2)
axes[1].axhline(80, color='gray', linestyle='--', lw=1, label='Seuil 80%')
axes[1].axhline(95, color='orange', linestyle='--', lw=1, label='Seuil 95%')
axes[1].fill_between(range(len(comps)), variance_cum, alpha=0.15, color='crimson')
axes[1].set_title('Variance cumulee (Coude)', fontweight='bold')
axes[1].set_ylabel('Variance cumulee (%)')
axes[1].set_xlabel('Composante principale')
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot7_scree_plot.png', dpi=150)
plt.show()

n_components_80 = np.argmax(variance_cum >= 80) + 1
print(f'=> {n_components_80} composantes suffisent pour expliquer 80% de la variance')

---
## 7. Integration NumPy + Pandas + Matplotlib : Exemples avances

In [ ]:
# Exemple 1 : Filtrage complexe avec NumPy dans Pandas
# Identifier les centrales anormalement grandes (> 2 ecarts-types)
mean_cap = np.mean(df['capacity_mw'].values)
std_cap  = np.std(df['capacity_mw'].values)
seuil    = mean_cap + 2 * std_cap

giants = df[df['capacity_mw'] > seuil][['name', 'country_long', 'primary_fuel', 'capacity_mw']]
giants = giants.sort_values('capacity_mw', ascending=False).head(10)

print(f'=== Top 10 centrales geantes (capacite > {seuil:.0f} MW = moy + 2*std) ===')
print(giants.to_string(index=False))

In [ ]:
# Exemple 2 : Scatter plot sophistique avec taille et couleur NumPy
df_scatter = df[df['primary_fuel'].isin(['Solar', 'Wind', 'Hydro', 'Coal', 'Gas'])].copy()
df_scatter = df_scatter[df_scatter['generation_gwh_2019'] > 0]

fuel_palette = {'Solar':'#FFD700', 'Wind':'#87CEEB',
                'Hydro':'#1E90FF', 'Coal':'#555', 'Gas':'#FF8C00'}

fig, ax = plt.subplots(figsize=(12, 7))

for fuel, color in fuel_palette.items():
    sub = df_scatter[df_scatter['primary_fuel'] == fuel]
    sizes = np.sqrt(sub['capacity_mw'].values) * 2
    ax.scatter(sub['capacity_mw'], sub['generation_gwh_2019'],
               c=color, s=sizes, alpha=0.5, label=fuel, edgecolors='none')

# Droite de regression NumPy
x_s = df_scatter['capacity_mw'].values
y_s = df_scatter['generation_gwh_2019'].values
coef = np.polyfit(x_s, y_s, 1)
x_line = np.linspace(x_s.min(), np.percentile(x_s, 98), 100)
ax.plot(x_line, np.polyval(coef, x_line), 'r--', lw=2, label=f'Regression (r={r:.2f})')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Capacite MW vs Generation GWh 2019 (log-log)', fontsize=14, fontweight='bold')
ax.set_xlabel('Capacite installee (MW) - log', fontsize=12)
ax.set_ylabel('Generation 2019 (GWh) - log', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot8_scatter_capacite_generation.png', dpi=150)
plt.show()

---
## Rapport de Synthese

In [ ]:
top1_pays  = top_cap.index[0]
top1_val   = top_cap.values[0]

rapport = f"""
============================================================
           RAPPORT DE SYNTHESE — WRI Global Power Plants
============================================================

DONNEES
-------
Nombre total de centrales analysees : {len(df):,}
Capacite totale mondiale             : {np.sum(df['capacity_mw'].values)/1000:.0f} GW
Pays representés                     : {df['country_long'].nunique()}
Types de carburant                   : {df['primary_fuel'].nunique()}

RESULTATS CLES
--------------
Pays avec plus de capacite : {top1_pays} ({top1_val:.0f} GW)
Carburant le plus repandu  : {fuel_count.index[0]} ({fuel_count.values[0]:,} centrales)
Carburant le plus puissant : {fuel_cap.index[0]} ({fuel_cap.values[0]:.0f} GW total)
Capacite moyenne / centrale: {np.mean(df['capacity_mw'].values):.1f} MW
Capacite mediane / centrale: {np.median(df['capacity_mw'].values):.1f} MW

STATISTIQUES (Test ANOVA)
-------------------------
F-stat = {f_stat:.2f}, p-value = {p_value:.2e}
=> La capacite differe significativement selon le carburant (p < 0.05)

TENDANCES TEMPORELLES
---------------------
Pente d'installation (1980-2023) : {coeffs[0]:+.1f} centrales/an
=> Construction en {'acceleration' if coeffs[0] > 0 else 'deceleration'}

ANALYSE MATRICIELLE (PCA)
-------------------------
{n_components_80} composantes principales suffisent
pour capturer 80% de la variance du dataset.

TENDANCES INTERESSANTES
-----------------------
1. La mediane de capacite (tres inferieure a la moyenne) indique
   que quelques mega-centrales tirent la moyenne vers le haut.
2. La carte geographique revele une concentration des centrales
   en Europe, Asie de l'Est et Est des Etats-Unis.
3. Les energies renouvelables (Solar, Wind) dominent en nombre
   mais le Hydro et le Coal restent leaders en capacite totale.
4. Forte correlation entre capacite installee et generation :
   r = {r:.3f} — les grandes centrales produisent proportionnellement plus.

OUTILS UTILISES
---------------
NumPy   : np.corrcoef, np.linalg.eig, np.polyfit, np.mean/std/median
Pandas  : groupby, value_counts, fillna, dropna, pivot
Matplotlib : bar, pie, scatter, boxplot, imshow
Seaborn : heatmap
SciPy   : stats.f_oneway (ANOVA), stats.pearsonr
============================================================
"""
print(rapport)